In [0]:
library(dataiku)
library(stats) # need this to calculate Mahalanobis Distance
library(parallel) # parallelize
library(dplyr)
library(FNN)
library(cluster)

In [0]:
# Recipe inputs
counterfactual_test_data <- dkuReadDataset("counterfactual_test_data", samplingMethod="head", nbRows=100000)
hurdle_predictions_testing <- dkuManagedFolderPath("5NPBmWH1")

In [0]:
colnames(counterfactual_test_data)

In [0]:
# get unique municipality observations
mun_properties  <- counterfactual_test_data %>% 
    distinct(Mun_Code, 
             blue_ss_frac,
             blue_ls_frac,
             red_ls_frac,
             orange_ls_frac,
             yellow_ss_frac,
             red_ss_frac,
             orange_ss_frac,
             yellow_ls_frac,
             roof_strong_wall_strong,
             roof_strong_wall_light,
             roof_strong_wall_salv,
             roof_light_wall_strong,
             roof_light_wall_light,
             roof_light_wall_salv,
             roof_salv_wall_strong,
             roof_salv_wall_light,
             roof_salv_wall_salv,
             island_groups,
             .keep_all = FALSE)

In [0]:
# variables I'm interested in for matching:
match_vars  <- c('blue_ss_frac',
                    'blue_ls_frac',
                    'red_ls_frac',
                    'orange_ls_frac',
                    'yellow_ss_frac',
                    'red_ss_frac',
                    'orange_ss_frac',
                    'yellow_ls_frac',
                    'roof_strong_wall_strong',
                    'roof_strong_wall_light',
                    'roof_strong_wall_salv',
                    'roof_light_wall_strong',
                    'roof_light_wall_light',
                    'roof_light_wall_salv',
                    'roof_salv_wall_strong',
                    'roof_salv_wall_light',
                    'roof_salv_wall_salv'
                   )

In [0]:
# Normalize the variables using z-score
mun_scaled <- mun_properties %>%
  mutate(across(c(blue_ss_frac:roof_salv_wall_salv), scale))

In [0]:
head(mun_scaled)

In [0]:
# Add a column for group labels (if not already present)
# Assuming `group` column exists and contains 3 groups (1, 2, 3)

# Split dataset by group
group1 <- mun_scaled %>% filter(island_groups == "Luzon")
group2 <- mun_scaled %>% filter(island_groups == "Visayas")
group3 <- mun_scaled %>% filter(island_groups == "Mindanao")

# Ensure only numeric columns are used for matching
group1_data <- group1 %>% select(-Mun_Code, -island_groups)
group2_data <- group2 %>% select(-Mun_Code, -island_groups)
group3_data <- group3 %>% select(-Mun_Code, -island_groups)

# Find nearest neighbors from group1 to group2
nn1_2 <- get.knnx(data = group2_data, query = group1_data, k = 1)

# Find nearest neighbors from group1 to group3
nn1_3 <- get.knnx(data = group3_data, query = group1_data, k = 1)

# Merge results
matches <- data.frame(
  Group1_ID = group1$Mun_Code, 
  Closest_Group2_ID = group2$Mun_Code[nn1_2$nn.index[,1]], 
  Closest_Group3_ID = group3$Mun_Code[nn1_3$nn.index[,1]]
)

# Print results
print(matches)

In [0]:
nrow(matches)

In [0]:
# Check group sizes
cat("Group Sizes: Luzon =", nrow(group1), " Visayas =", nrow(group2), " Mindanao =", nrow(group3), "\n")

# Ensure only numeric columns are used
group1_data <- group1 %>% select(-Mun_Code, -island_groups)
group2_data <- group2 %>% select(-Mun_Code, -island_groups)
group3_data <- group3 %>% select(-Mun_Code, -island_groups)

# Find nearest neighbors
nn1_2 <- get.knnx(data = group2_data, query = group1_data, k = 1)
nn1_3 <- get.knnx(data = group3_data, query = group1_data, k = 1)

# Ensure indices do not exceed available rows
nn1_2_indices <- pmin(nn1_2$nn.index[,1], nrow(group2))
nn1_3_indices <- pmin(nn1_3$nn.index[,1], nrow(group3))

# Merge results
matches <- data.frame(
  Group1_ID = group1$Mun_Code, 
  Closest_Group2_ID = group2$Mun_Code[nn1_2_indices], 
  Closest_Group3_ID = group3$Mun_Code[nn1_3_indices]
)

# Print results
print(dim(matches))  # Should match group1's size
print(head(matches))

In [0]:
mun_properties %>%
    filter(Mun_Code %in% c("PH015501000", "PH072245000", "PH126303000"))

In [0]:
# Creating a function that generates a counterfactual dataset

#' @title counterfactual_gen
#' @description Function takes argguments of df, tc, matches and returns a 
#' counterfactual dataframe to be used by the hurdle function
#' @param df counterfactual_test_data
#' @param tc tropical cyclone name
#' @param matches municipality matches
#' @return counterfactual_data_list return 

counterfactual_gen  <- function(df, tc, matches){
    
    # get unique municipality codes
    mun_code  <- unique(df$Mun_Code)
    
    # filter df by tc and get hazard characheristics
    counterfactual_data  <- df %>%
        filter(typhoon == tc) %>% # keep the minimum distance from the filter
        mutate(track_min_dist = min(track_min_dist, na.rm = TRUE),
              rain_total = rain_total[which.min(track_min_dist)],
              wind_max = wind_max[which.min(track_min_dist)]
              )
    missing_mun  <- setdiff(df$Mun_Code, mun_code) # SHOULD BE AN IF statement about here
    
    # get the characteristics of the missing mun codes
    id  <- which(df$Mun_Code == missing_mun)
    
    remaining_mun  <- df %>%
        [id, -c("typhoon", "rain_total", "wind_max", "track_min_dist")]
    # populate the dataframe with the missing mun
    counterfactual_data
    
    # df should have all the 1478 municipalities
    
    return(counterfactual_data) # returns a dataframe (maybe list for more experiments)
}

In [0]:
melor_2015  <- counterfactual_gen(df = counterfactual_test_data, tc = "melor2015", matches)

head(melor_2015)

In [0]:
nrow(melor_2015)

In [0]:
# Recipe outputs
matching_counterfactuals <- dkuManagedFolderPath("ZO3oPxC1")

In [0]:
colnames(counterfactual_test_data)

In [0]:
unique(counterfactual_test_data$typhoon)